# 基本原理
The underlying principle of the method is the equalization, or whitening, of power in the à trous wavelet spectrum of theinput image at all scales and locations. An edge-avoiding modiﬁcation of the à trous transform that uses bilateral weighting by thelocal variance in the wavelet planes is used to suppress the undesirable halos otherwise produced by discontinuities in the data.

在所有位置上对a trous算法的每一级小波系数进行平均（白化，whitening）。同时使用了一种避免边缘效应的改进的a trous变换方法，通过在小波平面上采用基于局部方差的双边权重来抑制数据中的不连续性所产生的不必要的晕影。
# 优势
基本无需手动调参；可以自动在保留大尺度特征的情况下，突出显示小尺度的特征
# 方法
## 一、使用B3-spline函数作为尺度函数的atrous小波变换
### 1.尺度函数
$h_0 = \frac{1}{16} \left [1,4,6,4,1\right]$，$h_s$为在$h_0$的每两个数中插入$2^s$个0
### 2.各级小波系数
定义原图像为$c_0$，则$c_s$的递推关系为：
$$c_{s+1} = h_s \ast c_s $$
各级小波系数：
$$w_{s+1} = c_s - c_{s+1}$$
故有：
$$ c_0 = c_s + \sum_i^{s} w_s$$
## 二、白化
在将小波系数相加得到原始图像时，将小波系数在每一个像素上进行加权，使得每个尺度的局地功率都是归一的：
$$ \overline{c_0} = c_s + \sum_i^{s} \frac{w_s}{\sqrt{\overline{p}}}$$
其中，$\bar p = w_s^2 \ast h_S$
## 三、去噪
继续对小波系数在每一个像素上进行加权，加权系数用于去噪，与标准差、阈值、该点处的小波系数相关：
$$\overline{c_0} = c_s + \sum_i^{s} \alpha_s\left(n_s\sigma_s,w_s\right)\frac{w_s}{\sqrt{\overline{p}}}$$
### 1.高斯白噪声
由仪器的读取噪声和热噪声产生。令$\sigma_s = \sigma\sigma_s^1$，其中，$\sigma_s^1$为单位标准差的高斯白噪声在尺度s上的标准差
#### (1)硬阈值：
$$\alpha_s\left(n_s\sigma_s,w_s\right) =
\begin{cases}
1,w_s>n_s\sigma_s \\
0,w_s<n_s\sigma_s
\end{cases}
$$
硬阈值经常会在一些尖锐的边缘附近产生一些奇怪的“光晕”
#### (2)软阈值
$$\alpha_s\left(n_s\sigma_s,w_s\right) = erf\left(\frac{\left|w_s\right|}{n_s\sigma_s}\right)$$
### 2.泊松噪声
来自于光子散粒噪声(photon shot noise)。在每个像素处，$\sigma = \sqrt{gI+r^2}$，其中，g是仪器的光子放大倍率，即接收到一个光子产生的数据强度(DN)，r是仪器的读取噪声。
### 3.总去噪公式
在每一个像素处，对尺度s的小波系数：
$$\overline{c_0} = c_s + \sum_i^{s} \alpha_s\left(n_s\sigma_s,w_s\right)\frac{w_s}{\sqrt{\overline{p}}} = c_s+\sum_i^s \frac{erf\left(\frac{\left|w_s\right|}{n_s\sqrt{gI+r^2}\sigma_s^1}\right)}{\sqrt{\bar p}}$$
其中，$n_s$是给定的每一级小波系数的阈值，g是仪器的光子放大倍率，即接收到一个光子产生的数据强度(DN)，r是仪器的读取噪声，$\sigma_s^1$为单位标准差的高斯白噪声在尺度s上的标准差。
## 四、边缘感知变换(Edge-aware transform)
为了避免atrous小波变换在尖锐边缘处产生“光晕”，在卷积过程中增加一个双边滤波器(bilateral filtering)

原式：
$$c_{s+1}(k) = h_s(k) \ast c_s(k) = \sum_l c_s(k+2^sl)h(l)$$
增加双边滤波器：
$$c_{s+1}(k) = \frac{\sum_l c_s(k+2^sl)h(l)b(k,k+2^sl)}{\sum_l h(l)b(k,k+2^sl)}$$
其中，
$$b(k, k+2^sl)=exp\left(-\frac{1}{2}\left(\frac{c_s(k)-c_s(k+2^sl)}{\nu_s}\right)^2\right)$$
依据功率归一的思想，
$$\nu_s(k)^2 = h_s(k) \ast c_s(k)^2-\left(h_s(k) \ast c_s(k)\right)^2$$

# wow的使用
sunkit_image.enhance.wow(data, *, scaling_function=None, n_scales=None, weights=None, whitening=True, denoise_coefficients=None, noise=None, bilateral=None, bilateral_scaling=False, soft_threshold=True, preserve_variance=False, gamma=3.2, gamma_min=None, gamma_max=None, h=0)

## Parameters:
__data__: (numpy.ndarray 或者 sunpy.map.GenericMap) – 传递给函数的图像。

__scaling_function__: ({Triangle , B3spline}, string, 可选参数) – 用于atrous小波变换的尺度函数，来自watroo包。默认为B3spline。

__n_scales__: (int, 可选参数) – 用于小波变换的尺度数目。如果为None，则计算出与输入大小兼容的最大数量的尺度。默认为 None。

__weights__ (list of float, 可选参数) – 使用小波系数重构图像阶段中使用的可选重构权重。默认情况下，所有权重都设置为1。如果权重不等于1，则输出光谱将不为白噪声。默认为[]。

__whitening__: (bool) – 若为True（默认），将会使用白化算法。默认为True。

__denoise_coefficients__: (list of float, 可选参数) – Noise threshold, in units of the noise standard deviation, used at each scale to denoise the wavelet coefficients. Defaults to [].

__noise__: (numpy.ndarray or None, 可选参数) – 一张与输入数据形状相同的噪声标准差地图。可用于考虑空间相关的（即泊松）噪声。默认值为None。

__bilateral__ (int or None, 可选参数) – 使用双边滤波器形成边缘感知小波变换。双边滤波器可以避免在强梯度区域附近形成光晕。推荐的“自然”值为1。默认值为None。

__bilateral_scaling__: (bool, 可选参数) – 正在测试，请勿使用。默认为False。

__soft_threshold__: (bool, 可选参数) – 如果denoise_coefficients不为空，则不使用此参数。如果为True，则使用软阈值法进行去噪，否则使用硬阈值法。软阈值法通常能产生更少的晕影。默认值为True。

__preserve_variance__: (bool, 可选参数) – 正在测试，请勿使用。默认为False。

__gamma__: (float, 可选参数) – 用于计算全局伽马变换图像的值。默认值为3.2。

__gamma_min__: (float or None, 可选参数) – 伽马变换的最小输入。如果为None，则默认为数据的最小值。

__gamma_max__: (float or None, 可选参数) – 伽马变换的最大输入。如果为None，则默认为数据的最大值。

__h__: (float, 可选参数) – Weight of the gamma-scaled image wrt that of the filtered image. 默认为0。

## Returns:
numpy.ndarray or sunpy.map.GenericMap – Normalized image. If a map is input, a map is returned with new data and the same metadata.

In [2]:
import sunkit_image.enhance as enhance
import sunpy.map
import matplotlib.pyplot as plt
from dask.array.random import gamma
from matplotlib.colors import Normalize, LogNorm
import numpy as np
%matplotlib notebook

In [2]:
def draw_fig(sun_map, return_fig=False, **kwargs):
    if type(sun_map) == str:
        sun_map = sunpy.map.Map(sun_map)
    fig = plt.figure()
    ax = fig.add_subplot(projection=sun_map)
    im = sun_map.plot(axes=ax, **kwargs)
    if return_fig:
        return fig

对于hri数据，噪声通常是高频的，故一般出现在前几个尺度上。在wow介绍文章中，对hri数据的去噪，前三个尺度上使用5/2/1倍$\sigma$的阈值，其他尺度不进行去噪

In [8]:
example = sunpy.map.Map("./data/hrieuvopn/alignment_data/10moving/solo_L2_eui-hrieuvopn-image_20220330T043015227_V02.fits")
data = np.copy(example.data)
data = np.nan_to_num(data, nan=0.0)
example = sunpy.map.Map(data, example.meta)
whitening = enhance.wow(example)
denoise_gaussian_hard = enhance.wow(example, denoise_coefficients=[5, 2, 1], soft_threshold=False, gamma=1)
denoise_gaussian_soft = enhance.wow(example, denoise_coefficients=[5, 2, 1], soft_threshold=True, gamma=1)
bilateral = enhance.wow(example, denoise_coefficients=[5, 2, 1], soft_threshold=True, bilateral=1, gamma=1)

原始图像

In [7]:
draw_fig(example, cmap='sdoaia171', norm=Normalize())

<IPython.core.display.Javascript object>

白化后的图像

In [9]:
draw_fig(whitening, cmap='sdoaia171', norm=Normalize())

<IPython.core.display.Javascript object>

白化+去噪（高斯白噪声，硬阈值）

In [10]:
draw_fig(denoise_gaussian_hard, cmap='sdoaia171', norm=Normalize())

<IPython.core.display.Javascript object>

白化+去噪（高斯白噪声，软阈值）

In [11]:
draw_fig(denoise_gaussian_soft, cmap='sdoaia171', norm=Normalize())

<IPython.core.display.Javascript object>

白化+去噪（高斯白噪声，软阈值）+双边滤波器

In [12]:
draw_fig(bilateral, cmap='sdoaia171', norm=Normalize())

<IPython.core.display.Javascript object>

# 对时空图进行白化